# U8-01 · Equidad de un clasificador con los datos del censo

*Especialista en Inteligencia Artificial (IFCD107) · Unidad 8*

Ejecuta cada celda con **Mayús + Intro**. Todo funciona en el navegador; la primera vez tarda unos segundos en cargar las bibliotecas.

> Datos: Becker, B. y Kohavi, R. (1996). *Adult*. UCI, CC BY 4.0. https://doi.org/10.24432/C5XW20

## 8.2 · Medir la equidad

`w1_modelo.py`

In [ ]:
import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier
from adultos import d, X_e, X_p, y_e, y_p

print(len(d), "personas | ingresos altos:", f"{d.alto.mean():.1%}")
print(d.groupby("sexo").alto.mean().round(3).to_dict())
m = HistGradientBoostingClassifier(categorical_features="from_dtype",
                                   random_state=0)
cat = X_e.select_dtypes(exclude="number").columns
Xe, Xp = X_e.copy(), X_p.copy()
for c in cat:
    Xe[c] = Xe[c].astype("category")
    Xp[c] = pd.Categorical(Xp[c], categories=Xe[c].cat.categories)
m.fit(Xe, y_e)
print(f"acierto en la prueba: {m.score(Xp, y_p):.3f}")
pd.to_pickle((m, Xp), "modelo_adultos.pkl")

`w2_equidad.py`

In [ ]:
import pandas as pd
from adultos import X_p, y_p

m, Xp = pd.read_pickle("modelo_adultos.pkl")
p = m.predict(Xp)
r = pd.DataFrame({"sexo": X_p.sexo, "real": y_p, "pred": p})
def tasas(g):
    return pd.Series({
        "n": len(g),
        "predichos altos": g.pred.mean(),          # paridad
        "sensibilidad": g.pred[g.real == 1].mean(),  # oportunidad
        "falsos positivos": g.pred[g.real == 0].mean(),
        "precision": g.real[g.pred == 1].mean()})
t = r.groupby("sexo")[["real", "pred"]].apply(tasas).round(3)
print(t.to_string())
print("razón de predichos altos (mujer/hombre):",
      round(t.loc["Female", "predichos altos"]
            / t.loc["Male", "predichos altos"], 2))

## 8.3 · Quitar la variable sensible no basta

`w3_sin_sexo.py`

In [ ]:
import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier
from adultos import X_e, X_p, y_e, y_p

def preparar(X, ref):
    X = X.copy()
    for c in X.select_dtypes(exclude="number").columns:
        X[c] = pd.Categorical(X[c], categories=sorted(
            ref[c].dropna().unique()))
    return X
sin = ["sexo"]
Xe = preparar(X_e.drop(columns=sin), X_e)
Xp = preparar(X_p.drop(columns=sin), X_e)
m = HistGradientBoostingClassifier(categorical_features="from_dtype",
                                   random_state=0).fit(Xe, y_e)
p = m.predict(Xp)
for s in ["Female", "Male"]:
    print(f"sin la variable sexo, {s}: predichos altos"
          f" {p[X_p.sexo == s].mean():.3f}")
# ¿se puede deducir el sexo de las demás variables?
Xs = preparar(X_e.drop(columns=sin), X_e)
adivina = HistGradientBoostingClassifier(
    categorical_features="from_dtype", random_state=0)
adivina.fit(Xs, X_e.sexo)
print(f"acierto al deducir el sexo de las demás variables:"
      f" {adivina.score(Xp, X_p.sexo):.3f}")
print("proporción de mujeres en la prueba:",
      round((X_p.sexo == "Female").mean(), 3))

## 8.4 · Corregir: umbrales por grupo

`w4_umbrales.py`

In [ ]:
import numpy as np
import pandas as pd
from adultos import X_p, y_p

m, Xp = pd.read_pickle("modelo_adultos.pkl")
prob = m.predict_proba(Xp)[:, 1]
mujer = (X_p.sexo == "Female").to_numpy(); y = y_p.to_numpy()
def informe(umbral_m, umbral_h, nombre):
    p = np.where(mujer, prob >= umbral_m, prob >= umbral_h)
    sens = [p[g & (y == 1)].mean() for g in (mujer, ~mujer)]
    print(f"{nombre:<20} acierto {(p == y).mean():.3f}  sensibilidad"
          f" M {sens[0]:.3f} / H {sens[1]:.3f}")
informe(0.5, 0.5, "mismo umbral 0,5")
objetivo = np.mean(prob[~mujer & (y == 1)] >= 0.5)
u = np.quantile(prob[mujer & (y == 1)], 1 - objetivo)
print(f"umbral para mujeres que iguala la sensibilidad: {u:.3f}")
informe(u, 0.5, "umbrales distintos")

**Prueba.** Repite `w2_equidad.py` agrupando por raza en lugar de por sexo. ¿Qué métricas cambian más?